In [ ]:
# install from PyPI
!pip install openai

In [ ]:
import re
import pandas as pd
import json
import openai
import time
import os
from openai import OpenAI

In [ ]:
with open('C:/Users/User/Desktop/wanted_detail_improve_20250515.txt', 'r', encoding='utf-8') as f:
    wanted_data = json.load(f)

wanted_data_df = pd.DataFrame(wanted_data)
wanted_data_df = wanted_data_df.T

In [ ]:
wanted_data_df.drop(['회사명', '지역'], axis=1, inplace=True) # 회사명, 지역 필요없음

In [ ]:
wanted_data_df['신입 지원 가능 여부'].replace({True: '가능', False: '불가능'}, inplace=True)
wanted_data_df.head(3)

In [ ]:
wanted_data_df.head(9)

In [ ]:
title_list = []
wanted_data_list = []
for idx, row in wanted_data_df.iterrows():
    job_info = f"""
제목: {row['제목']}
회사소개: {row['회사 소개']}
주요업무: {row['주요 업무']}
자격요건: {row['자격 요건']}
우대사항: {row['우대 사항']}
복지: {row['복지']}
채용절차: {row['채용절차']}
장점: {row['장점']}
근무지: {row['근무지']}
직군: {row['직군']}
직무: {row['직무']}
기술스택: {row['기술 스택']}
요구최소경력: {row['요구 최소 경력']}년
요구최대경력: {row['요구 최대 경력']}년
신입지원가능여부: {row['신입 지원 가능 여부']}
마감일: {row['마감일']}
"""
    wanted_data_list.append(job_info)
    title_list.append(row['제목'])

In [ ]:
def toImproveText(text):
    if isinstance(text, list):  # 입력이 리스트인지 확인
        cleaned_list = []
        for item in text:
            cleaned_item = item.replace('\r', '').replace('\t', '')
            # +, #, :, %, ,는 유지하면서 나머지 특수문자 제거
            cleaned_item = re.sub(r'[^a-zA-Z0-9가-힣\s+#%:,]', '', cleaned_item).strip()
            cleaned_list.append(cleaned_item)
        return cleaned_list
    elif isinstance(text, int):
        return text
    else:
        cleaned_text = text.replace('\r', '').replace('\t', '')
        cleaned_text = re.sub(r'[^a-zA-Z0-9가-힣\s+#%:,]', '', cleaned_text).strip()
        return cleaned_text

In [ ]:
full_qa_list = []   # 전체 저장용
split_qa_list = []  # 2개마다 저장용
count = 1

for wanted_data_text in wanted_data_list:
    wanted_data_cleanedText = toImproveText(wanted_data_text)
    result = generate_qa_pairs(wanted_data_cleanedText)

    full_qa_list.append(result)     # 전체용
    split_qa_list.append(result)    # 500개마다 저장용
    
    print(f"==================== {count}번째 완료 ====================")
    print(f"원문:\n{wanted_data_text}")
    print(f"{result}")

    if count % 500 == 0:
        # 💾 500개마다 저장 코드
        # split_qa_list만 가지고 저장하고
        # 저장 후 초기화만 split_qa_list만!
        qa_dict = {}
        for i in range(500):
            result = split_qa_list[i]
            title = title_list[count - 500 + i]  # 현재 인덱스 계산
            qa_pairs = re.findall(r"Q\d+:\s*(.*?)\nA\d+:\s*(.*?)(?=\nQ\d+:|\Z)", result, re.DOTALL)

            qa_items = []
            for q, a in qa_pairs:
                qa_items.append({
                    "질문": q.strip(),
                    "답변": a.strip()
                })

            qa_dict[title] = qa_items

        with open(f'qa_set_20250515_{count}.json', 'w', encoding='utf-8') as f:
            json.dump(qa_dict, f, ensure_ascii=False, indent=2)
        print(f"✅ qa_set_20250515_{count}.json 저장 완료")

        # csv 파일로 저장
        csv_rows = []
        
        for title, qa_list in qa_dict.items():
            for qa in qa_list:
                csv_rows.append({
                    "제목": title,
                    "질문": qa["질문"],
                    "답변": qa["답변"]
                })
        
        qa_set_df = pd.DataFrame(csv_rows)
        qa_set_df.to_csv(f"qa_set_20250515_{count}.csv", index=False, encoding='utf-8-sig')
        print(f"✅ qa_set_20250515_{count}.csv 저장 완료")
        split_qa_list = []  # 🔁 2개 저장 후 초기화만 부분 리스트!
    count += 1

In [ ]:
# 마지막에 전체 저장
qa_dict = {}

for i in range(len(full_qa_list)):
    result = full_qa_list[i]
    qa_pairs = re.findall(r"Q\d+:\s*(.*?)\nA\d+:\s*(.*?)(?=\nQ\d+:|\Z)", result, re.DOTALL)

    qa_items = []
    for q, a in qa_pairs:
        qa_items.append({
            "질문": q.strip(),
            "답변": a.strip()
        })

    qa_dict[title_list[i]] = qa_items

with open('qa_set_20250515_all.json', 'w', encoding='utf-8') as f:
    json.dump(qa_dict, f, ensure_ascii=False, indent=2)

print("✅ 전체 JSON 저장 완료: qa_set_20250515_all.json")


In [ ]:
# csv 파일로 저장
csv_rows = []

for title, full_qa_list in qa_dict.items():
    for qa in full_qa_list:
        csv_rows.append({
            "제목": title,
            "질문": qa["질문"],
            "답변": qa["답변"]
        })

qa_set_df = pd.DataFrame(csv_rows)
qa_set_df.to_csv("qa_set_20250515_all.csv", index=False, encoding='utf-8-sig')
print("✅ 전체 csv 저장 완료: qa_set_20250515_all.csv")